# PRIXON — Qwen2.5-7B-Instruct QLoRA starter

This notebook trains a **separate PRIXON LoRA adapter**. It does not change the upstream Qwen checkpoint, tokenizer, architecture or original chat template. The first run is limited to identity, conversational style and contextual language.

**Do not use private or sensitive personal/computer data.** The tiny repository seed dataset is for validating the data pipeline only and is below the minimum training threshold.

In [ ]:
%pip install -q "transformers>=4.56,<5" "trl>=0.22,<1" "peft>=0.15,<1" "bitsandbytes>=0.45,<1" "datasets>=3,<5" "accelerate>=1,<2" "huggingface_hub>=0.29,<1"
 

In [ ]:
import sys
import json
import hashlib
import subprocess
import platform
import zipfile
from pathlib import Path
from datetime import datetime, timezone
import torch

REPO_URL = "https://github.com/AryanGupta-234/P1-.git"
REPO_BRANCH = "dev"
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
SEED = 2026
MAX_SEQ_LENGTH = 2048
MIN_TRAIN_EXAMPLES = 100
MIN_TOTAL_EXAMPLES = 125
RECOMMENDED_TRAIN_EXAMPLES = 500
ALLOWED_TRAINING_CATEGORIES = {"identity", "conversation_style", "contextual_language"}
KAGGLE_DATASET_FILE = Path("/kaggle/input/prixon-training-data/seed_v1.jsonl")
OUTPUT_DIR = Path("/kaggle/working/prixon_run")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("A GPU accelerator is required. In Kaggle, open Notebook Settings and select a GPU.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print("Python:", platform.python_version(), "PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)


In [ ]:
repo_dir = Path("/kaggle/working/P1-")
if not repo_dir.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(repo_dir)], check=True)
sys.path.insert(0, str(repo_dir))

from aie.prixon.dataset import read_jsonl, prepare_dataset  # noqa: E402

source_file = KAGGLE_DATASET_FILE
if not source_file.is_file():
    source_file = repo_dir / "datasets" / "prixon" / "seed_v1.jsonl"
    print("Using repository seed data for pipeline validation only:", source_file)
records = read_jsonl(source_file)
eligible = [row for row in records if row["category"] in ALLOWED_TRAINING_CATEGORIES]
print("Validated examples:", len(records), "Eligible initial-adapter examples:", len(eligible))
if len(eligible) < MIN_TOTAL_EXAMPLES:
    raise RuntimeError(
        f"Only {len(eligible)} eligible samples. Add at least {MIN_TOTAL_EXAMPLES} reviewed examples to produce "
        f"at least {MIN_TRAIN_EXAMPLES} training samples after the 80/10/10 split; "
        "500+ is recommended for the first meaningful experiment. The eight-example seed is not a training corpus."
    )
# Split only the phase-one categories. Specialist/general examples must not inflate
# the initial adapter sample count or silently enter its training partitions.
eligible_source = OUTPUT_DIR / "phase_one_eligible.jsonl"
with eligible_source.open("w", encoding="utf-8") as handle:
    for row in eligible:
        clean = {key: value for key, value in row.items() if key != "conversation_sha256"}
        handle.write(json.dumps(clean, ensure_ascii=False, sort_keys=True) + "\n")

prepared_dir = OUTPUT_DIR / "dataset"
manifest = prepare_dataset(eligible_source, prepared_dir, seed=SEED, overwrite=True)
print(json.dumps(manifest, indent=2))
 

## Load train, validation and held-out test partitions

The initial adapter is intentionally prevented from seeing reasoning or specialist categories. The held-out test partition is never passed to the trainer.

In [ ]:
from datasets import load_dataset

data_files = {name: str(prepared_dir / f"{name}.jsonl") for name in ("train", "validation", "test")}
raw = load_dataset("json", data_files=data_files)

def eligible(row):
    return row["category"] in ALLOWED_TRAINING_CATEGORIES

raw = {name: split.filter(eligible) for name, split in raw.items()}
if len(raw["train"]) < MIN_TRAIN_EXAMPLES:
    raise RuntimeError("The train partition has fewer than the required eligible examples.")

def to_prompt_completion(row):
    messages = row["messages"]
    if not messages or messages[-1]["role"] != "assistant":
        raise ValueError(f"Invalid assistant completion in {row.get('id')}")
    return {"prompt": messages[:-1], "completion": [messages[-1]]}

train_ds = raw["train"].map(to_prompt_completion, remove_columns=raw["train"].column_names)
validation_ds = raw["validation"].map(to_prompt_completion, remove_columns=raw["validation"].column_names)
test_ds = raw["test"]
print({name: len(value) for name, value in raw.items()})
print("First prompt-completion sample:", train_ds[0])


## Pin the exact foundation revision

Resolve the upstream commit SHA at the start of this run and record it. For reproducibility, later reruns should set BASE_REVISION to the recorded SHA instead of resolving a moving branch.

In [ ]:
from huggingface_hub import model_info
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

BASE_REVISION = model_info(BASE_MODEL).sha
print("Foundation revision:", BASE_REVISION)
if not BASE_REVISION or len(BASE_REVISION) != 40:
    raise RuntimeError("Could not resolve an immutable foundation commit SHA.")

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_REVISION, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    revision=BASE_REVISION,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map="auto",
)
model.config.use_cache = False
print("Tokenizer chat template present:", bool(tokenizer.chat_template))
print("EOS token:", repr(tokenizer.eos_token))


## Capture baseline generations before training

These are qualitative comparison artifacts, not a complete capability benchmark. Keep the test partition held out. Inspect the paired generations manually before promotion.

In [ ]:
def generate_answer(messages, max_new_tokens=256):
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH)
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = output[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

def collect_generations(rows, label):
    results = []
    for row in rows:
        messages = row["messages"][:-1]
        reference = row["messages"][-1]["content"]
        results.append({
            "id": row["id"], "category": row["category"], "reference": reference,
            "generation": generate_answer(messages), "stage": label,
        })
    return results

baseline_results = collect_generations(test_ds, "foundation_baseline")
baseline_path = OUTPUT_DIR / "baseline_generations.jsonl"
baseline_path.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in baseline_results), encoding="utf-8")
print("Baseline cases captured:", len(baseline_results))

regression_file = repo_dir / "datasets" / "prixon" / "regression_cases_v1.jsonl"
regression_cases = [json.loads(line) for line in regression_file.read_text(encoding="utf-8").splitlines() if line.strip()]
def collect_preservation_regressions(cases, stage):
    outputs = []
    for case in cases:
        answer = generate_answer([{"role": "user", "content": case["prompt"]}], max_new_tokens=384)
        outputs.append({"id": case["id"], "category": case["category"], "critical": case["critical"],
                        "prompt": case["prompt"], "rubric": case["rubric"], "generation": answer, "stage": stage})
    return outputs
baseline_regressions = collect_preservation_regressions(regression_cases, "foundation_baseline")
baseline_regression_path = OUTPUT_DIR / "baseline_preservation_regressions.jsonl"
baseline_regression_path.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in baseline_regressions), encoding="utf-8")
print("Foundation preservation cases captured:", len(baseline_regressions))


## Attach LoRA and run QLoRA supervised fine-tuning

The foundation parameters remain frozen. Only the adapter parameters are trained and saved. The first experiment uses conservative settings and does not merge the adapter.

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
import importlib.metadata as metadata

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)
training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR / "trainer_checkpoints"),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    warmup_ratio=0.03,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    optim="paged_adamw_8bit",
    bf16=(compute_dtype == torch.bfloat16),
    fp16=(compute_dtype == torch.float16),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    completion_only_loss=True,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=validation_ds if len(validation_ds) else None,
    peft_config=lora_config,
    processing_class=tokenizer,
)
train_result = trainer.train()
print(train_result)

# Evaluate through the adapter-wrapped model, not the frozen base reference.
model = trainer.model
model.eval()
 

## Save adapter-only artifact and compare generations

Do not export a merged model in this first run. The output is an adapter candidate tied to the exact foundation revision recorded below.

In [ ]:
from pathlib import Path

adapter_dir = OUTPUT_DIR / "prixon_adapter"
adapter_dir.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(adapter_dir, safe_serialization=True)

post_results = collect_generations(test_ds, "prixon_adapter_candidate")
post_path = OUTPUT_DIR / "adapter_generations.jsonl"
post_path.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in post_results), encoding="utf-8")

adapter_regressions = collect_preservation_regressions(regression_cases, "prixon_adapter_candidate")
adapter_regression_path = OUTPUT_DIR / "adapter_preservation_regressions.jsonl"
adapter_regression_path.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in adapter_regressions), encoding="utf-8")

packages = {}
for package in ("torch", "transformers", "trl", "peft", "bitsandbytes", "datasets", "accelerate", "huggingface_hub"):
    try:
        packages[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        packages[package] = "not-installed"

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

adapter_files = sorted(p for p in adapter_dir.rglob("*") if p.is_file())
manifest_out = {
    "artifact_type": "peft_lora_adapter_only",
    "foundation_model": BASE_MODEL,
    "foundation_revision": BASE_REVISION,
    "foundation_weights_modified": False,
    "tokenizer_model": BASE_MODEL,
    "tokenizer_revision": BASE_REVISION,
    "original_chat_template_retained": True,
    "training_method": "QLoRA 4-bit NF4",
    "training_categories": sorted(ALLOWED_TRAINING_CATEGORIES),
    "dataset_manifest": manifest,
    "packages": packages,
    "gpu": torch.cuda.get_device_name(0),
    "compute_dtype": str(compute_dtype),
    "adapter_files": {str(p.relative_to(adapter_dir)): sha256_file(p) for p in adapter_files},
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "promotion_status": "candidate_not_approved",
    "evaluation_note": "Generation pairs require human review; this run is not a complete capability regression benchmark.",
}
manifest_path = OUTPUT_DIR / "prixon_manifest.json"
manifest_path.write_text(json.dumps(manifest_out, indent=2, ensure_ascii=False), encoding="utf-8")

zip_path = Path("/kaggle/working/prixon_adapter_candidate.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in adapter_files:
        archive.write(path, path.relative_to(adapter_dir))
    archive.write(manifest_path, "prixon_manifest.json")
    archive.write(baseline_path, "baseline_generations.jsonl")
    archive.write(post_path, "adapter_generations.jsonl")
    archive.write(baseline_regression_path, "baseline_preservation_regressions.jsonl")
    archive.write(adapter_regression_path, "adapter_preservation_regressions.jsonl")
print("Adapter candidate:", zip_path)
print("Manifest:", manifest_path)


## Promotion checklist

- [ ] Verify the foundation revision and tokenizer revision are the expected pinned values.
- [ ] Inspect adapter generations beside the foundation baseline.
- [ ] Run a separate held-out benchmark for reasoning, coding, maths, planning, structured outputs, Windows internals, networking, cybersecurity and troubleshooting.
- [ ] Check contextual-language improvement without teaching the model to ignore AIE action authorization.
- [ ] Check privacy, factuality, instruction hierarchy and output-format regressions.
- [ ] Keep the adapter disabled by default until reviewed and approved.

**This notebook has not been run by the repository authoring process.** A created ZIP is a candidate artifact, not a validated release.
- [ ] Compare the paired foundation/adapter generations for the 12 preserved-capability regression cases; review each rubric manually.
